# Ingest Drivers JSON File

This notebook demonstrates the ingestion of the drivers.json file into the bronze layer.

## What this notebook covers
* Load the drivers.json source file
* Define and enforce schema (preserve the nested structure)
* Add ingestion metadata columns: source_file and ingestion_timestamp
* Write the transformed data to the bronze Delta table
* Validate the loaded results

This uses centralized configuration and helper functions for consistency.

In [0]:
%run ../00-common/01.Formula1_Configuration_Setup

In [0]:
# Check configured catalog
catalog_name

In [0]:
%run ../00-common/02.bronze-helper

In [0]:
# Check landing folder path
landing_folder_path

In [0]:
# Define source file and target table
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
# Preview source file path
source_file

In [0]:
# define the schema
from pyspark.sql.types import StructType, StructField,StringType,DateType
name_schema = StructType([
    StructField("givename", StringType()),
    StructField("familyname", StringType())
])

drivers_schema = StructType([
    StructField("driverId", StringType()),
    StructField("name",name_schema),
    StructField("dateOfBirth", DateType()),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])

In [0]:
# Read drivers JSON into a DataFrame
drivers_df = (
    spark.read
    .format("json")
    .schema(drivers_schema)
    .option("multiLine", True)
    .option("mode", "FAILFAST")
    .load(source_file)
)

display(drivers_df)

In [0]:
# Add ingestion metadata columns
drivers_final_df = add_ingestion_metadata(drivers_df)
display(drivers_final_df)

In [0]:
# Write drivers data to the bronze Delta table
(
    drivers_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_name)
)

In [0]:
# Preview target table name
table_name

In [0]:
%sql
-- Query the bronze drivers table
SELECT * FROM formula1.bronze.drivers

In [0]:
# Display the bronze drivers table
display(spark.table(table_name))